# Create a fresh YOLO detection split

Create a new seeded random split of the current images, independent of campaigns,
previous splits, and the migration manifest. Defaults: 80% train, 10% validation,
and the remaining 10% test. Related patches may therefore occur in different
splits; this measures performance on randomly held-out images, not unseen campaigns.

Images and labels stay in place. The notebook writes `train.txt`, `val.txt`,
`test.txt`, `data.yaml`, and `split_summary.json` into the dataset directory.
The detection labels contain only class `0: tobacco`; semantic masks are not used.
Run after finishing dataset preparation. Keep the seed and image inventory fixed
to reproduce the split. Regenerate the YAML after moving the dataset to another machine.


In [5]:
from pathlib import Path
import json
import math
import random

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "manual_notebooks").is_dir() and (path / "data").is_dir()
)


In [6]:
DATASET = ROOT / "data" / "tobacco_migrated"
SEED = 42
TRAIN_FRACTION = 0.01
VAL_FRACTION = 0.01
CLASS_NAMES = {0: "tobacco"}
RUN_SPLIT = True
OVERWRITE = True


In [7]:
def prepare_split(dataset, seed, train_fraction, val_fraction, class_names):
    """Validate paired detection data and build a fresh image-level split."""
    dataset = Path(dataset).resolve()
    if not (0 < train_fraction < 1 and 0 < val_fraction < 1
            and train_fraction + val_fraction < 1):
        raise ValueError("Train, validation, and test fractions must be positive and sum to 1.")
    if not class_names or set(class_names) != set(range(len(class_names))):
        raise ValueError("Class IDs must be consecutive integers starting at zero.")
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    images = sorted(path for path in (dataset / "images").iterdir()
                    if path.is_file() and path.suffix.lower() in extensions)
    if len({path.stem for path in images}) != len(images):
        raise ValueError("Image stems must be unique for matching label files.")
    empty_labels = 0
    for image in images:
        label = dataset / "labels" / f"{image.stem}.txt"
        if not label.is_file():
            raise FileNotFoundError(f"Missing label: {label}")
        lines = label.read_text(encoding="utf-8").splitlines()
        empty_labels += not any(line.strip() for line in lines)
        for number, line in enumerate(lines, 1):
            if not line.strip():
                continue
            fields = line.split()
            if len(fields) != 5:
                raise ValueError(f"Expected five detection fields: {label}:{number}")
            class_id, *box = map(float, fields)
            if (not math.isfinite(class_id) or class_id not in class_names
                    or not all(math.isfinite(x) and 0 <= x <= 1 for x in box)
                    or box[2] <= 0 or box[3] <= 0):
                raise ValueError(f"Invalid detection label: {label}:{number}")
    random.Random(seed).shuffle(images)
    train_count = int(len(images) * train_fraction)
    val_count = int(len(images) * val_fraction)
    splits = {
        "train": images[:train_count],
        "val": images[train_count:train_count + val_count],
        "test": images[train_count + val_count:],
    }
    if any(not paths for paths in splits.values()):
        raise ValueError("Too few images for nonempty splits with these fractions.")
    summary = {
        "method": "seeded_random_images",
        "seed": seed,
        "requested_fractions": {"train": train_fraction, "val": val_fraction,
                                "test": 1 - train_fraction - val_fraction},
        "counts": {name: len(paths) for name, paths in splits.items()},
        "empty_label_files": empty_labels,
        "images": {name: [p.name for p in paths] for name, paths in splits.items()},
    }
    return splits, summary


def save_split(dataset, splits, summary, class_names, overwrite=False):
    """Write split lists and YAML without moving images or labels."""
    dataset = Path(dataset).resolve()
    payloads = {
        f"{name}.txt": "".join(f"./images/{p.name}\n" for p in paths)
        for name, paths in splits.items()
    }
    payloads["data.yaml"] = (
        f"path: {json.dumps(dataset.as_posix())}\n"
        "train: train.txt\nval: val.txt\ntest: test.txt\nnames:\n"
        + "".join(f"  {key}: {json.dumps(value)}\n" for key, value in class_names.items())
    )
    payloads["split_summary.json"] = json.dumps(summary, indent=2) + "\n"
    existing = [name for name in payloads if (dataset / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Split outputs already exist: {existing}. Set OVERWRITE=True to replace them.")
    for name, content in payloads.items():
        (dataset / name).write_text(content, encoding="utf-8")


In [8]:
if RUN_SPLIT:
    splits, summary = prepare_split(
        DATASET, SEED, TRAIN_FRACTION, VAL_FRACTION, CLASS_NAMES
    )
    save_split(DATASET, splits, summary, CLASS_NAMES, overwrite=OVERWRITE)
    print(json.dumps(summary["counts"], indent=2))
    print("Empty label files:", summary["empty_label_files"])
    print("Dataset YAML:", DATASET.resolve() / "data.yaml")
else:
    print("Split disabled. Set RUN_SPLIT=True to create the split and data.yaml.")


{
  "train": 25,
  "val": 25,
  "test": 2458
}
Empty label files: 9
Dataset YAML: /home/natsusa/academic/tobacco-article/data/tobacco_migrated/data.yaml


## Train using this split

Set `RUN_SPLIT = True` and run all cells. For 2,508 images, the default fractions
produce 2,006 training, 250 validation, and 252 test images (rounding remainder
is assigned to test). Existing split outputs require `OVERWRITE = True` to replace.

Then open `manual_notebooks/models_training.ipynb`. Its shared dataset path points
to `data/tobacco_migrated/data.yaml`, with task `detect`. Choose training parameters
before running it. Existing trained weights retain the usual skip behavior.
The test split is reserved for final evaluation; training uses train and validation.
